# 인증 CORS 타임아웃

Bearer 인증, 브라우저 출처 허용, 응답 시간 상한을 각각 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 기본 실행

허용한 출처의 사전 요청과 올바른 인증 헤더를 보냅니다.

In [2]:
import asyncio
from fastapi.testclient import TestClient
from agent_service.app import app, get_agent, get_timeout
client = TestClient(app)
preflight = client.options('/v1/chat', headers={
    'Origin': 'http://localhost:5173',
    'Access-Control-Request-Method': 'POST',
    'Access-Control-Request-Headers': 'Authorization,Content-Type',
})
print(preflight.status_code, preflight.headers['access-control-allow-origin'])
assert preflight.status_code == 200

200 http://localhost:5173


## 인증 실패 관찰

토큰이 없으면 Agent 호출 전에 401이 반환됩니다.

In [3]:
missing = client.post('/v1/chat',
    json={'question': '질문', 'thread_id': 'auth-1'})
print(missing.status_code, missing.json())
assert missing.status_code == 401

401 {'detail': '인증 정보가 올바르지 않습니다'}


## 시간 초과 관찰

지연 Agent와 짧은 시간 상한을 주입해 단일 응답과 스트림의 차이를 확인합니다.

In [4]:
class SlowAgent:
    async def ainvoke(self, state):
        await asyncio.sleep(0.05)
        return {'answer': '늦은 답'}
    async def astream(self, state, stream_mode):
        await asyncio.sleep(0.05)
        yield {'token': '늦은 답'}

app.dependency_overrides[get_agent] = lambda: SlowAgent()
app.dependency_overrides[get_timeout] = lambda: 0.01
headers = {'Authorization': 'Bearer local-demo-token'}

In [5]:
single_timeout = client.post('/v1/chat', headers=headers,
    json={'question': '질문', 'thread_id': 'slow-1'})
stream_timeout = client.post('/v1/chat/stream', headers=headers,
    json={'question': '질문', 'thread_id': 'slow-2'})
print(single_timeout.status_code, 'event: error' in stream_timeout.text)
assert single_timeout.status_code == 504
assert stream_timeout.status_code == 200
assert 'event: error' in stream_timeout.text
app.dependency_overrides.clear()

Agent SSE stream failed
Traceback (most recent call last):
  File "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/agent_service/app.py", line 110, in chat_stream
    async for chunk in runtime.astream(
  File "/var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/ipykernel_40078/1613430500.py", line 6, in astream
    await asyncio.sleep(0.05)
  File "[사용자 폴더]/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/tasks.py", line 665, in sleep
    return await future
           ^^^^^^^^^^^^
asyncio.exceptions.CancelledError

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/agent_service/app.py", line 109, in chat_stream
    async with asyncio.timeout(timeout):
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "[사용자 폴더]/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/timeouts.py"

504 True


## 실패 재현

허용 목록에 없는 출처는 CORS 허용 헤더를 받지 못합니다.

In [6]:
blocked = client.options('/v1/chat', headers={
    'Origin': 'https://unknown.example',
    'Access-Control-Request-Method': 'POST',
})
print(blocked.status_code, blocked.headers.get('access-control-allow-origin'))
assert 'access-control-allow-origin' not in blocked.headers

400 None


## 수정

정확한 출처와 인증 헤더를 함께 보내면 요청이 성공합니다. 운영에서는 환경별 허용 목록을 관리합니다.

In [7]:
ok = client.post('/v1/chat',
    headers={'Authorization': 'Bearer local-demo-token'},
    json={'question': '질문', 'thread_id': 'auth-1'})
print(ok.status_code, ok.json()['answer'])
assert ok.status_code == 200

200 질문을 확인했습니다: 질문
